### Install Dependencies

In [20]:
import os
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain.chat_models import init_chat_model
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

### Load Environment Variables

In [21]:
load_dotenv()

True

### Temporary Directory

In [22]:
tmpdir = "../tmp/"

### GPT Models

In [23]:
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [24]:
llm = init_chat_model(model="gpt-4o-mini", temperature=0.2)

### Indexing

#### Load Document

In [25]:
KNOWLEDGE_BASE = """# LangChain Framework

LangChain is a framework for developing applications powered by language models. It was created by Harrison Chase in October 2022.

## Core Components

1. **Models**: LangChain supports various LLM providers including OpenAI, Anthropic, and local models.

2. **Prompts**: Templates for structuring inputs to language models.

3. **Chains**: Sequences of calls to models and other components.

4. **Agents**: Systems that use LLMs to determine which actions to take.

5. **Memory**: Components for persisting state between chain/agent calls.

## LangGraph

LangGraph is a library for building stateful, multi-actor applications. Key features:
- State management
- Cycles and loops
- Human-in-the-loop
- Persistence

## Pricing

LangChain itself is open source and free. LangSmith (the observability platform) has a free tier and paid plans starting at $39/month.

## Getting Started

Install with: pip install langchain langchain-openai
Create your first chain in under 10 lines of code.
"""

In [26]:
doc = Document(
    page_content=KNOWLEDGE_BASE, metadata={"source": "langchain_knowledge_base.md"}
)

#### Chunking

In [27]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

In [28]:
chunks = splitter.split_documents([doc])

#### Create Vector Store

In [29]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_model,
    collection_name="test_collection",
    persist_directory=os.path.abspath(tmpdir)
)

### Retrieval Augmented Generation (RAG)

#### Retriever

In [30]:
retriever = vectorstore.as_retriever(
    search_type="similarity", search_kwargs={"k": 2}
)

In [31]:
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

#### RAG Prompt Template

In [32]:
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:

{context}

Question: {question}

Answer:


Make sure to answer in a concise manner, 
and if you don't know the answer, just say "I don't know."""
)

#### RAG Chain

In [33]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

### Test RAG Chain

In [34]:
question = "What is LangChain?"
answer = rag_chain.invoke(question)
answer

'LangChain is a framework for developing applications powered by language models, created by Harrison Chase in October 2022. It includes core components such as models, prompts, chains, agents, and memory.'

In [35]:
question = "Who created LangChain?"
answer = rag_chain.invoke(question)
answer

'LangChain was created by Harrison Chase.'

In [36]:
question = "What is LangGraph used for?"
answer = rag_chain.invoke(question)
answer

'LangGraph is used for building stateful, multi-actor applications, focusing on state management, cycles and loops, human-in-the-loop interactions, and persistence.'

In [37]:
question = "What is Electromagnetism?"
answer = rag_chain.invoke(question)
answer

"I don't know."

### Cleanup ChromaDB Colelction

In [38]:
import chromadb
import os

chroma_client = chromadb.PersistentClient(
    path=os.path.abspath(tmpdir)
)

print(chroma_client.list_collections())

chroma_client.delete_collection(
    name="test_collection"
)

[Collection(name=test_collection)]
